In [170]:
import pandas as pd

movies = pd.read_csv('movies2.csv')

In [5]:
 #Evaluation accuracy of 66% (vs. random-chance baseline of 14%)
from transformers import pipeline
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base",top_k = None, device = "cpu")
classifier("I love this!")

Device set to use cpu


[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528691716492176},
  {'label': 'neutral', 'score': 0.005764589179307222},
  {'label': 'anger', 'score': 0.004419787786900997},
  {'label': 'sadness', 'score': 0.002092392183840275},
  {'label': 'disgust', 'score': 0.001611991785466671},
  {'label': 'fear', 'score': 0.0004138525982853025}]]

In [47]:
import re

def check_sentence(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text)
                 if s.strip()]

In [153]:
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
movieId = []
emotion_scores = {label: [] for label in emotion_labels}


def calc_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    
    for prediction in predictions:  # iterate over sentences
        for label in emotion_labels:
            # find the score for this label in the sentence predictions
            score = next((x["score"] for x in prediction if x["label"] == label), 0)
            per_emotion_scores[label].append(score)
    
    # pick the max per emotion across sentences
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [158]:
from tqdm import tqdm



for i in tqdm(range(len(movies))):
    movieId.append(movies["movie_id"][i])
    sentences = check_sentence(movies["description"][i])
    predictions = classifier(sentences)
    max_scores = calc_max_emotion_scores(predictions)
    
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 7556/7556 [07:44<00:00, 16.27it/s]


In [160]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["movie_id"] = movieId
emotions_df

,anger,disgust,fear,joy,sadness,surprise,neutral,movie_id
0,0.010945,0.016719,0.015639,0.049233,0.040791,0.052863,0.813809,1
1,0.026336,0.011352,0.008016,0.011586,0.006201,0.014004,0.922506,2
2,0.010945,0.016719,0.015639,0.049233,0.040791,0.052863,0.813809,1
3,0.026336,0.011352,0.008016,0.011586,0.006201,0.014004,0.922506,2
4,0.007151,0.021208,0.943477,0.003916,0.017488,0.001159,0.005600,3
...,...,...,...,...,...,...,...,...
7553,0.017623,0.007971,0.143528,0.726576,0.173834,0.038766,0.738560,7907
7554,0.095467,0.218491,0.030795,0.132243,0.132106,0.294606,0.653838,7909
7555,0.125225,0.119891,0.023659,0.052945,0.017963,0.029951,0.885090,7910
7556,0.009575,0.915959,0.033246,0.009019,0.018079,0.001822,0.012299,7911


In [166]:
#drop duplicates from emotions_df
emotions_df = emotions_df.drop_duplicates(subset=["movie_id"], keep='first')
emotions_df = emotions_df.reset_index(drop=True)
emotions_df

,anger,disgust,fear,joy,sadness,surprise,neutral,movie_id
0,0.010945,0.016719,0.015639,0.049233,0.040791,0.052863,0.813809,1
1,0.026336,0.011352,0.008016,0.011586,0.006201,0.014004,0.922506,2
2,0.007151,0.021208,0.943477,0.003916,0.017488,0.001159,0.005600,3
3,0.010933,0.005289,0.970210,0.000839,0.004172,0.002980,0.005577,4
4,0.039122,0.086691,0.014777,0.011955,0.028363,0.010856,0.867076,5
...,...,...,...,...,...,...,...,...
7551,0.017623,0.007971,0.143528,0.726576,0.173834,0.038766,0.738560,7907
7552,0.095467,0.218491,0.030795,0.132243,0.132106,0.294606,0.653838,7909
7553,0.125225,0.119891,0.023659,0.052945,0.017963,0.029951,0.885090,7910
7554,0.009575,0.915959,0.033246,0.009019,0.018079,0.001822,0.012299,7911


In [172]:
movies = pd.merge(movies, emotions_df, on="movie_id")

In [175]:
movies.to_csv("movies_with_emotions.csv", index=False)